In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [3]:
import os
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")

In [6]:
embed = embeddings.embed_query("Hello AI")

In [7]:
len(embed)

384

In [8]:
embed = embeddings.embed_query("Hello AI, this is genai concepts")

In [9]:
len(embed)

384

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
documents = [
             "What is the capital of USA?",
             "Who is the president of USA?",
             "WHo is prime minister of India?"
            ]

In [12]:
my_query = "Narendra Modi is the prime minister of India"

In [13]:
document_embeddings = embeddings.embed_documents(documents)

In [14]:
len(document_embeddings)

3

In [15]:
len(document_embeddings[0]) # ---> first stateent

384

In [16]:
len(document_embeddings[1])

384

In [17]:
len(document_embeddings[2])

384

In [18]:
query_embeddings = embeddings.embed_query(my_query)

In [19]:
len(query_embeddings)

384

In [20]:
cosine_similarity([query_embeddings],document_embeddings)

array([[0.06192698, 0.31278137, 0.73973244]])

In [21]:
my_query = "Leader of the countries"
query_embeddings = embeddings.embed_query(my_query)
cosine_similarity([query_embeddings],document_embeddings)

array([[0.27063273, 0.5998015 , 0.57583515]])

In [22]:
from sklearn.metrics.pairwise import euclidean_distances

In [23]:
euclidean_distances([query_embeddings], document_embeddings)

array([[1.20778083, 0.8946491 , 0.92104816]])

| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


In [24]:
import faiss
import chromadb
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.docstore.in_memory import InMemoryDocstore





C:\Users\DELL\AppData\Local\Temp\ipykernel_3576\2203642273.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS, Chroma


In [25]:
# Index first


index = faiss.IndexFlatL2(384)

In [26]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000013F1C323CC0> >

In [26]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id = {},
)

In [27]:
vector_store

In [29]:
vector_store.add_texts(["AI is future", "AI is powerful", "Dogs are cute", "Pizza is good", "I like mexican food"])

['29d3e9ac-1cac-482f-ba92-9d50f2a42582',
 'f679fd84-7631-4dd2-bb55-5b49022938f9',
 '0eb60080-9230-44d5-b617-d9beefe92e06',
 '07c44894-afba-47e7-8fce-27fe6ed3e2df',
 'e1c6fa35-6a89-44a4-b81a-fdfe7f2a79fc']

In [30]:
vector_store.index_to_docstore_id

{0: '29d3e9ac-1cac-482f-ba92-9d50f2a42582',
 1: 'f679fd84-7631-4dd2-bb55-5b49022938f9',
 2: '0eb60080-9230-44d5-b617-d9beefe92e06',
 3: '07c44894-afba-47e7-8fce-27fe6ed3e2df',
 4: 'e1c6fa35-6a89-44a4-b81a-fdfe7f2a79fc'}

In [31]:
faiss_index = 1
docstore_id = vector_store.index_to_docstore_id[faiss_index]
print(docstore_id)


f679fd84-7631-4dd2-bb55-5b49022938f9


In [32]:
vector_store.docstore.search(docstore_id)

Document(id='f679fd84-7631-4dd2-bb55-5b49022938f9', metadata={}, page_content='AI is powerful')

In [33]:
results = vector_store.similarity_search("Tell me about AI",k=2)

# Tell me about AI ---> embeddings (embedding model)
# Configurations in Index --> similarity
# Ordr them and filter them and give us k documents

In [34]:
results

[Document(id='f679fd84-7631-4dd2-bb55-5b49022938f9', metadata={}, page_content='AI is powerful'),
 Document(id='29d3e9ac-1cac-482f-ba92-9d50f2a42582', metadata={}, page_content='AI is future')]

In [35]:
results = vector_store.similarity_search("I love cat",k=2)

In [36]:
results

[Document(id='0eb60080-9230-44d5-b617-d9beefe92e06', metadata={}, page_content='Dogs are cute'),
 Document(id='e1c6fa35-6a89-44a4-b81a-fdfe7f2a79fc', metadata={}, page_content='I like mexican food')]

In [37]:
results = vector_store.similarity_search("I love cat",k=5)

In [38]:
results

[Document(id='0eb60080-9230-44d5-b617-d9beefe92e06', metadata={}, page_content='Dogs are cute'),
 Document(id='e1c6fa35-6a89-44a4-b81a-fdfe7f2a79fc', metadata={}, page_content='I like mexican food'),
 Document(id='07c44894-afba-47e7-8fce-27fe6ed3e2df', metadata={}, page_content='Pizza is good'),
 Document(id='f679fd84-7631-4dd2-bb55-5b49022938f9', metadata={}, page_content='AI is powerful'),
 Document(id='29d3e9ac-1cac-482f-ba92-9d50f2a42582', metadata={}, page_content='AI is future')]

In [39]:
results = vector_store.similarity_search("My favourite super hero",k=5)
results

[Document(id='f679fd84-7631-4dd2-bb55-5b49022938f9', metadata={}, page_content='AI is powerful'),
 Document(id='0eb60080-9230-44d5-b617-d9beefe92e06', metadata={}, page_content='Dogs are cute'),
 Document(id='29d3e9ac-1cac-482f-ba92-9d50f2a42582', metadata={}, page_content='AI is future'),
 Document(id='e1c6fa35-6a89-44a4-b81a-fdfe7f2a79fc', metadata={}, page_content='I like mexican food'),
 Document(id='07c44894-afba-47e7-8fce-27fe6ed3e2df', metadata={}, page_content='Pizza is good')]

| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


In [28]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [41]:
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [60]:
index = faiss.IndexFlatIP(384)
vector_store = FAISS(
    embedding_function=embeddings,
    index= index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [61]:
vector_store.add_documents(documents = documents)

['1b7728c1-33b9-4604-bc01-31bdce06c46d',
 '800098cd-3693-4e88-8327-747060b37b19',
 '354834fc-5e2c-4a45-bf3d-318b55268d1c',
 '42003747-83dd-45a9-9889-fa8edf0c5115',
 '70ce78a2-dba8-44d2-8d4c-a648dfc27973',
 '6b04ad76-1f1f-42a2-8d05-d51c08286808',
 'c5456ded-bdd7-47af-a439-83dd53308332',
 'f7975f28-9cc6-4508-9bdb-c2c88f903230',
 '6aa0d251-de86-44cb-b2b6-104bb33fae09',
 'c59a514b-53f5-4b5b-9cc7-a3cbfeb278fd']

In [62]:
vector_store.add_texts(["AI is future", "AI is powerful", "Dogs are cute", "Pizza is good", "I like mexican food"])

['7a36738b-c853-4c1c-ab2f-5bd61f78645c',
 '1806aedd-3af3-416e-812c-78373dcf2071',
 'bc0c0f6c-628a-42d2-92c4-267b8fc53fbf',
 '716e42e6-0b36-45ca-bcbb-93d1be56866e',
 '944f6afb-4e7e-49a4-b908-0c7f666bb98c']

In [69]:
vector_store.index_to_docstore_id

{0: '1b7728c1-33b9-4604-bc01-31bdce06c46d',
 1: '800098cd-3693-4e88-8327-747060b37b19',
 2: '354834fc-5e2c-4a45-bf3d-318b55268d1c',
 3: '42003747-83dd-45a9-9889-fa8edf0c5115',
 4: '70ce78a2-dba8-44d2-8d4c-a648dfc27973',
 5: '6b04ad76-1f1f-42a2-8d05-d51c08286808',
 6: 'c5456ded-bdd7-47af-a439-83dd53308332',
 7: 'f7975f28-9cc6-4508-9bdb-c2c88f903230',
 8: '6aa0d251-de86-44cb-b2b6-104bb33fae09',
 9: 'c59a514b-53f5-4b5b-9cc7-a3cbfeb278fd',
 10: '7a36738b-c853-4c1c-ab2f-5bd61f78645c',
 11: '1806aedd-3af3-416e-812c-78373dcf2071',
 12: 'bc0c0f6c-628a-42d2-92c4-267b8fc53fbf',
 13: '716e42e6-0b36-45ca-bcbb-93d1be56866e',
 14: '944f6afb-4e7e-49a4-b908-0c7f666bb98c'}

In [64]:
vector_store.similarity_search("Langchain provides abstractions to make working with LLMs easy",
                               k=2) #Hyperparameter

[Document(id='354834fc-5e2c-4a45-bf3d-318b55268d1c', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='f7975f28-9cc6-4508-9bdb-c2c88f903230', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [65]:
vector_store.similarity_search("Which animal is cute?",
                               k=2) #Hyperparameter

[Document(id='bc0c0f6c-628a-42d2-92c4-267b8fc53fbf', metadata={}, page_content='Dogs are cute'),
 Document(id='944f6afb-4e7e-49a4-b908-0c7f666bb98c', metadata={}, page_content='I like mexican food')]

In [66]:
vector_store.similarity_search(
    "Langchain provides abstractions to make working with LLMs easy",
    k=4,
    filter = {"source" : {"$eq": "news"}}
)

[Document(id='42003747-83dd-45a9-9889-fa8edf0c5115', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='800098cd-3693-4e88-8327-747060b37b19', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='6aa0d251-de86-44cb-b2b6-104bb33fae09', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]

In [67]:
vector_store.similarity_search_with_score("Langchain provides abstractions to make working with LLMs easy",
    k=4)

[(Document(id='354834fc-5e2c-4a45-bf3d-318b55268d1c', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
  np.float32(0.57917786)),
 (Document(id='f7975f28-9cc6-4508-9bdb-c2c88f903230', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
  np.float32(0.49078575)),
 (Document(id='7a36738b-c853-4c1c-ab2f-5bd61f78645c', metadata={}, page_content='AI is future'),
  np.float32(0.16697761)),
 (Document(id='1806aedd-3af3-416e-812c-78373dcf2071', metadata={}, page_content='AI is powerful'),
  np.float32(0.13363138))]

In [68]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs = {
        "score_threshold" : 0.001,
        })

In [70]:
vector_store.save_local("BIA FAISS Langchain")

In [71]:
new_vector_store = FAISS.load_local("BIA FAISS Langchain", embeddings,allow_dangerous_deserialization=True)

In [73]:
new_vector_store.similarity_search("Langchain")

[Document(id='354834fc-5e2c-4a45-bf3d-318b55268d1c', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='f7975f28-9cc6-4508-9bdb-c2c88f903230', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='7a36738b-c853-4c1c-ab2f-5bd61f78645c', metadata={}, page_content='AI is future'),
 Document(id='944f6afb-4e7e-49a4-b908-0c7f666bb98c', metadata={}, page_content='I like mexican food')]

In [57]:
vector_store_2 = FAISS.from_documents(
    documents, 
    embeddings, 
    distance_strategy="COSINE" # explicit cosine strategy
)



In [50]:
vector_store.as_retriever?

Signature: vector_store.as_retriever(**kwargs: 'Any') -> 'VectorStoreRetriever'
Docstring:
Return `VectorStoreRetriever` initialized from this `VectorStore`.

Args:
    **kwargs: Keyword arguments to pass to the search function.

        Can include:

        * `search_type`: Defines the type of search that the Retriever should
            perform. Can be `'similarity'` (default), `'mmr'`, or
            `'similarity_score_threshold'`.
        * `search_kwargs`: Keyword arguments to pass to the search function.

            Can include things like:

            * `k`: Amount of documents to return (Default: `4`)
            * `score_threshold`: Minimum relevance threshold
                for `similarity_score_threshold`
            * `fetch_k`: Amount of documents to pass to MMR algorithm
                (Default: `20`)
            * `lambda_mult`: Diversity of results returned by MMR;
                `1` for minimum diversity and 0 for maximum. (Default: `0.5`)
            * `filter`:

In [80]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [37]:
retrieved_documents = vector_store.similarity_search_with_score("Langchain provides abstractions to make working with LLMs easy",
    k=15)

In [40]:
r_doc = retrieved_documents[0]
len(r_doc)

2

In [45]:
r_doc

(Document(id='cd1720f8-3b26-4d75-bb66-213d4e75fb95', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 np.float32(0.57917786))

In [44]:
r_doc[1]

np.float32(0.57917786)

In [41]:
r_doc[0]

Document(id='cd1720f8-3b26-4d75-bb66-213d4e75fb95', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!')

In [48]:
# Above 60%

doc_lis = []
thresh = 0.5
for r_doc in retrieved_documents:
    score = r_doc[1]
    if score >= thresh:
        doc_lis.append(r_doc[0])
    

In [49]:
doc_lis

[Document(id='cd1720f8-3b26-4d75-bb66-213d4e75fb95', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!')]

In [65]:
vector_store.similarity_search_with_relevance_scores("Langchain provides abstractions to make working with LLMs easy",
    k=4)

[(Document(id='889739a6-feec-484b-8620-f9120ea997d0', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
  np.float32(0.5904594)),
 (Document(id='541c128e-3102-4a3d-b725-805bf0c0cb40', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
  np.float32(0.6529621)),
 (Document(id='3af99412-84f4-448b-8fc4-0b2739f0b8d7', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
  np.float32(0.9794073)),
 (Document(id='c973396f-1bc3-4a14-8f8e-a363a1767ee0', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
  np.float32(0.99073684))]

In [81]:
retriever.invoke("Langchain provides abstractions to make working with LLMs easy")

[Document(id='889739a6-feec-484b-8620-f9120ea997d0', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='541c128e-3102-4a3d-b725-805bf0c0cb40', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [64]:
# Invoke the retriever
docs = retriever.invoke("LangChain projects")

for doc in docs:
    print(f"Content: {doc.page_content}")
    # Check if 'score' or 'relevance' exists in the metadata
    print(f"Metadata: {doc.metadata}")

Content: I have a bad feeling I am going to get deleted :(
Metadata: {'source': 'tweet'}
Content: The top 10 soccer players in the world right now.
Metadata: {'source': 'website'}


# Chroma

In [59]:
!pip install langchain-chroma chromadb

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
from langchain_chroma import Chroma

In [55]:
vector_store = Chroma.from_documents(
    documents = documents,
    embedding=embeddings,
    collection_name = "l2_collection",
    persist_directory="./chroma_l2_db",
    collection_metadata = {"hnsw:space" : "l2"}
)

In [57]:
retriever = vector_store.as_retriever(search_kwargs = {"k":2})

In [58]:
retriever.invoke("Langchain provides abstractions to make working with LLMs easy")

[Document(id='dcd4e039-acda-4327-bc55-f9044af3c1ee', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='b7284a02-2d04-4350-9046-69c3286cc470', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]